# Fitting the Crab with Fermi/LAT, HAWC, VERITAS and H.E.S.S.

Based on a Notebook created by Anjana Kaushik Talluri

## Goal

Perform a joint Fit using those four instruments, while VERITAS and H.E.S.S. use the GammpyLike plugin

## Steps
1. First setup a shared Pointsource Model for the Crab
2. Load and prepare the different data
3. Create a BayesianAnalaysis instance and use MultiNest to explore the Posterior
4. Take a look at the results

# Import packages

First import all the packages we will need in the following

In [ ]:
import astropy.units as u
from threeML import *
from threeML.io.package_data import get_path_of_data_file
import warnings
import numpy as np
import shutil
from IPython.display import Image, display
import glob
from pathlib import Path
import matplotlib as mpl
from matplotlib import pyplot as plt
from astropy.io import fits as pyfits
import scipy as sp
import hawc_hal
from threeML import *
import os
import fermipy
from gammapy_plugin.GammapyLike import GammapyLike
import yaml

In [ ]:
from gammapy.datasets.map import MapDataset
from gammapy.datasets import Datasets
from gammapy.data.data_store import DataStore
from pathlib import Path
from astropy.coordinates import Angle, SkyCoord
from regions import CircleSkyRegion
import matplotlib.pyplot as plt

from gammapy.data import DataStore
from gammapy.datasets import (
    Datasets,
    FluxPointsDataset,
    SpectrumDataset,
    SpectrumDatasetOnOff,
)
from gammapy.estimators import FluxPointsEstimator
from gammapy.estimators.utils import resample_energy_edges
from gammapy.makers import (
    ReflectedRegionsBackgroundMaker,
    SafeMaskMaker,
    SpectrumDatasetMaker,
)
from gammapy.maps import MapAxis, RegionGeom, WcsGeom
from gammapy.modeling import Fit
from gammapy.modeling.models import (
    ExpCutoffPowerLawSpectralModel,
    SkyModel,
    create_crab_spectral_model,
)


# Define a common model of a point source with a LogParabola spectrum

In [ ]:
spectrum = Log_parabola()


spectrum.piv.value = 1
spectrum.piv.fix = True
spectrum.piv.unit = u.TeV

spectrum.K.value = 3.37e-20

spectrum.K.prior = Log_uniform_prior(lower_bound = 10e-30,upper_bound = 10e-3)
spectrum.K.min_value = 10e-50

spectrum.alpha.prior = Uniform_prior(lower_bound = -10,upper_bound =10)
spectrum.beta.prior = Uniform_prior(lower_bound = -10,upper_bound =10)

source = PointSource("crab", ra=83.63, dec=22.01 , spectral_shape=spectrum)
#testing a bunch of things
#smg = SpectralModelGenerator(spectrum)
#gpy_p = smg.class_def()

In [ ]:
type(spectrum.alpha.min_value)

In [ ]:

f_model = Model(source)
f_model.display()
#ene = np.array([1,10,100])



# VERITAS

Let's load the VERITAS data

In [ ]:

with open("/home/tobi/sw/gammapy-plugin/examples/config.yaml","r") as f:
    config_file = yaml.safe_load(f)
data_dir = config_file['data']['anasum']
output_dir = config_file['fileio']['outdir']
on_region_radius = Angle(
    "{} deg".format(np.sqrt(config_file['cuts']['th2cut']))
)
emin = config_file['selection']['emin']
emax = config_file['selection']['emax']
nbin = config_file['selection']['nbin']
exclusion_on = config_file['selection']['exc_on_region_radius']
exc_radius = config_file['selection']['exc_radius']
datastore = DataStore.from_dir(data_dir)
obs_table = datastore.obs_table
obs_ids = obs_table['OBS_ID']
available_irf = ["aeff", "edisp"]
observations = datastore.get_observations(
    obs_ids, required_irf=available_irf
)
RA = obs_table['RA_OBJ']
DEC = obs_table['DEC_OBJ']
RA_OBJ = RA[0]
DEC_OBJ = DEC[0]
target_position = SkyCoord(
    ra=RA_OBJ, dec=DEC_OBJ, unit="deg", frame="icrs"
)
on_region_radius = Angle("{} deg".format(np.sqrt(0.008)))
on_region = CircleSkyRegion(
    center=target_position, radius=on_region_radius
)

exclusion_mask = []

reg0 = CircleSkyRegion(
    center=SkyCoord(RA_OBJ, DEC_OBJ, unit="deg", frame="icrs"),
    radius= exclusion_on * u.deg,
)
reg1 = CircleSkyRegion(
    center=SkyCoord(81.9087, 21.937, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg2 = CircleSkyRegion(
    center=SkyCoord(82.6806, 22.4623, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg3 = CircleSkyRegion(
    center=SkyCoord(83.4118, 20.4742, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg4 = CircleSkyRegion(
    center=SkyCoord(84.1099, 21.9931, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg5 = CircleSkyRegion(
    center=SkyCoord(84.4112, 21.1425, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg6 = CircleSkyRegion(
    center=SkyCoord(84.8629, 21.7629, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg7 = CircleSkyRegion(
    center=SkyCoord(85.4782, 23.3262, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)
reg8 = CircleSkyRegion(
    center=SkyCoord(85.5166, 22.6603, unit="deg", frame="icrs"),
    radius= exc_radius * u.deg,
)

skydir = target_position.galactic
geom = WcsGeom.create(
    npix=(1000, 1000),
    binsz=0.005,
    skydir=skydir,
    proj="TAN",
    frame="icrs",
)
exclusion_mask = ~geom.region_mask(
    [reg0, reg1, reg2, reg3, reg4, reg5, reg6, reg7, reg8]
)

energy_ax = MapAxis.from_energy_bounds(
    1e-2,
    1e4,
    nbin= nbin,
    per_decade=True,
    unit="TeV",
    name="energy",
)
energy_ax_true = MapAxis.from_energy_bounds(
    1e-2,
    1e4,
    nbin= nbin,
    per_decade=True,
    unit="TeV",
    name="energy_true",
)

geom = RegionGeom.create(region=on_region, axes=[energy_ax])
dataset_empty = SpectrumDataset.create(
    geom=geom, energy_axis_true=energy_ax_true
)

dataset_maker = SpectrumDatasetMaker(
    containment_correction=False,
    selection=["counts", "exposure", "edisp"],
)
bkg_maker = ReflectedRegionsBackgroundMaker(
    exclusion_mask=exclusion_mask
)

safe_mask_masker = SafeMaskMaker(
    methods=["aeff-max"], aeff_percent=10
)

datasets = Datasets()
for obs_id, observation in zip(obs_ids, observations):
    dataset = dataset_maker.run(
        dataset_empty.copy(name=str(obs_id)), observation
    )
    dataset_on_off = bkg_maker.run(dataset, observation)
    dataset_on_off = safe_mask_masker.run(
        dataset_on_off, observation
    )
    datasets.append(dataset_on_off)


In [ ]:
VERITAS = GammapyLike("Veritas")
VERITAS.set_datasets(datasets)

# FERMI

In [ ]:
lat_catalog = FermiLATSourceCatalog()

ra, dec, table = lat_catalog.search_around_source("Crab", radius=0.004)

table

In [ ]:
model = lat_catalog.get_model()

In [ ]:
model.display()

In [ ]:
model.free_point_sources_within_radius(3.0, normalization_only=True)
model.Crab_IC.spectrum.main.Log_parabola.K.prior = Log_uniform_prior(lower_bound = 10e-30,upper_bound=10e3)
model.Crab_synch.spectrum.main.Log_parabola.K.prior = Log_uniform_prior(lower_bound = 10e-30,upper_bound=10e3)
model.display()

In [ ]:
ra = 83.63
dec = 22.01

In [ ]:
# Download data from Jan 01 2010 to February 1 2010

tstart = "2010-01-01 00:00:00"
tstop = "2010-02-01 00:00:00"

# Note that this will understand if you already download these files, and will
# not do it twice unless you change your selection or the outdir

evfile, scfile = download_LAT_data(
    ra,
    dec,
    20.0,
    tstart,
    tstop,
    time_type="Gregorian",
    destination_directory="Crab_data",
)

In [ ]:
config = FermipyLike.get_basic_config(
    evfile=evfile,
    scfile=scfile,
    ra=ra,
    dec=dec,
    fermipy_verbosity=1,
    fermitools_chatter=0,
)

# See what we just got

config.display()

In [ ]:
LAT = FermipyLike("LAT", config)
config.display()

# HAWC

In [ ]:
from hawc_hal import HAL, HealpixConeROI

In [ ]:
import requests
import shutil
import os


def get_hawc_file(filename, odir="./", overwrite=False):

    if overwrite or not os.path.exists(odir + filename):
        url = "https://data.hawc-observatory.org/datasets/crab_data/public_data/crab_2017/"

        req = requests.get(url + filename, verify=False, stream=True)
        req.raw.decode_content = True
        with open(odir + filename, "wb") as f:
            shutil.copyfileobj(req.raw, f)

    return odir + filename


maptree = "HAWC_9bin_507days_crab_data.hd5"
response = "HAWC_9bin_507days_crab_response.hd5"
odir = "./"


maptree = get_hawc_file(maptree, odir)
response = get_hawc_file(response, odir)

assert os.path.exists(maptree)
assert os.path.exists(response)


In [ ]:
# Define the ROI.
ra_crab, dec_crab = 83.63, 22.02
data_radius = 3.0  # in degree
model_radius = 3.0  # in degree

roi = HealpixConeROI(
    data_radius=data_radius, model_radius=model_radius, ra=ra_crab, dec=dec_crab
)

# Instance the plugin
hawc = HAL("HAWC", maptree, response, roi, flat_sky_pixels_size=0.1)

# Use from bin 1 to bin 9
hawc.set_active_measurements(1, 9)

In [ ]:
hawc.display()

In [ ]:
# Load the H.E.S.S. public Data Release from the GAMMAPY_DATA folder
ds = DataStore.from_dir(os.path.join(os.environ.get("GAMMAPY_DATA"),"hess-dl3-dr1/"))

In [ ]:
selected_tab = ds.obs_table.select_sky_circle(SkyCoord.from_name("crab"),2*u.deg)
obs_ids = selected_tab["OBS_ID"]
obs = ds.get_observations(obs_ids)
target_position = SkyCoord(ra=83.63, dec=22.01, unit="deg", frame="icrs")
on_region_radius = Angle("0.11 deg")
on_region = CircleSkyRegion(center=target_position, radius=on_region_radius)
exclusion_region = CircleSkyRegion(
    center=SkyCoord(183.604, -8.708, unit="deg", frame="galactic"),
    radius=0.5 * u.deg,
)

skydir = target_position.galactic
geom = WcsGeom.create(
    npix=(150, 150), binsz=0.05, skydir=skydir, proj="TAN", frame="icrs"
)

exclusion_mask = ~geom.region_mask([exclusion_region])
exclusion_mask.plot()
plt.show()
energy_axis = MapAxis.from_energy_bounds(
    0.1, 40, nbin=20, per_decade=True, unit="TeV", name="energy"
)
energy_axis_true = MapAxis.from_energy_bounds(
    0.05, 100, nbin=30, per_decade=True, unit="TeV", name="energy_true"
)

geom = RegionGeom.create(region=on_region, axes=[energy_axis])
dataset_empty = SpectrumDataset.create(geom=geom, energy_axis_true=energy_axis_true)

dataset_maker = SpectrumDatasetMaker(
    containment_correction=True, selection=["counts", "exposure", "edisp"]
)
bkg_maker = ReflectedRegionsBackgroundMaker(exclusion_mask=exclusion_mask)
safe_mask_maker = SafeMaskMaker(methods=["aeff-max"], aeff_percent=10)

In [ ]:
datasets_hess = Datasets()

for obs_id, observation in zip(obs_ids, obs):
    dataset = dataset_maker.run(dataset_empty.copy(name=str(obs_id)), observation)
    dataset_on_off = bkg_maker.run(dataset, observation)
    dataset_on_off = safe_mask_maker.run(dataset_on_off, observation)
    datasets_hess.append(dataset_on_off)
datasets_hess

In [ ]:
HESS = GammapyLike("Hess")
HESS.set_datasets(datasets_hess)

In [ ]:
ba = BayesianAnalysis(f_model,DataList(VERITAS,LAT,hawc,HESS))

In [ ]:
ba.set_sampler("multinest")
ba.sampler.setup()

# Need to reset the priors after setting up everything for some reason
f_model.LAT_galdiff_Prefactor.prior = Uniform_prior(lower_bound= 0.1,upper_bound =10)
f_model.LAT_isodiff_Normalization.prior = Log_uniform_prior(lower_bound = 10e-3,upper_bound =10e2)

This may now take some time, you might want to grab a coffee:)

In [ ]:
ba.sample()

In [ ]:
results = ba.results

In [ ]:
# currently only works for Fermi/LAT
display_spectrum_model_counts(ba)

# Some HAWC plots

In [ ]:
hawc.display_fit(smoothing_kernel_sigma=0.01, display_colorbar=True)

In [ ]:
hawc.display_spectrum()

In [ ]:
fig, ax = plt.subplots()
plot_spectra(
    results,
    ene_min=1.0,
    ene_max=37,
    num_ene=50,
    energy_unit="TeV",
    flux_unit="TeV/(s cm2)",
    subplot=ax,
)
ax.set_xlim(0.8, 100)
ax.set_ylabel(r"$E^2\,dN/dE$ [TeV cm$^{-2}$ s$^{-1}$]")
ax.set_xlabel("Energy [TeV]")

In [ ]:
fig